# Hurst Pairs Alpha Hunt — 2026-06-02

**Hypothesis** (from `strategy_engine2/output/hurst-optimal-exit-pairs/spec.json`): when a crypto pair's log-spread is anti-persistent (local generalized Hurst `H < 0.5`) and the spread sits 1-2σ from its rolling mean, the spread mean-reverts fast. Per-trade O-U fit calibrates profit-take and stop-loss thresholds; 72h vertical barrier.

**Framing** (option C): spec ported into a stateful custom `signal_construction` module that emits per-coin signals. Pipeline wraps it; `rank` / `decay` / most of `turnover_control` skipped (would smear discrete entries/exits).

**Universe:** top-50 Binance USDT spot pairs by trailing 30-day dollar ADV, point-in-time. USD- and gold-pegged tokens excluded.

**Data:** `backtest_engine2/data/binance_hourly_top50_pool.parquet` — hourly close + USDT `quote_volume`, 66 candidate symbols, 2019-01-01 → 2026-05-21.

**Annualisation:** `ANN_FACTOR = 24 × 365 = 8760` (hourly, 24/7 crypto).

## Cell 1 — Stage 1: Universe + data load

Skill: `backtest_engine2/skills/13-universe.md`. Load the long-format parquet, pivot to wide date×asset frames, drop pegged tokens, build the top-50 point-in-time mask via 30-day trailing dollar ADV.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(r"C:\Users\User\backtest_engine")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from backtest.alpha_pipeline import universe

DATA_DIR = REPO_ROOT / "backtest_engine2" / "data"
ANN_FACTOR = 24 * 365  # 8760, hourly 24/7

# --- Load --------------------------------------------------------------
raw = pd.read_parquet(DATA_DIR / "binance_hourly_top50_pool.parquet")
print(f"raw shape: {raw.shape}")
print(f"raw time range: {raw['open_time'].min()} -> {raw['open_time'].max()}")
print(f"raw n_symbols: {raw['symbol'].nunique()}")

# --- Exclude pegged tokens ---------------------------------------------
USD_PEGGED  = {"USD1USDT", "USDSUSDT", "RLUSDUSDT", "BFUSDUSDT", "USDEUSDT"}
GOLD_PEGGED = {"XAUTUSDT", "PAXGUSDT"}
EXCLUDE = USD_PEGGED | GOLD_PEGGED
filtered = raw[~raw["symbol"].isin(EXCLUDE)].copy()
print(
    f"after dropping {len(EXCLUDE)} pegged tokens "
    f"({sorted(EXCLUDE)}): {filtered['symbol'].nunique()} candidates remain"
)

# --- Pivot long -> wide ------------------------------------------------
prices        = filtered.pivot(index="open_time", columns="symbol", values="close")
dollar_volume = filtered.pivot(index="open_time", columns="symbol", values="quote_volume")
# Strip timezone to a naive UTC index (downstream modules expect this)
prices.index        = pd.DatetimeIndex(prices.index).tz_convert(None)
dollar_volume.index = prices.index
prices        = prices.sort_index()
dollar_volume = dollar_volume.sort_index()

print(f"\nwide prices:        shape={prices.shape}")
print(f"wide dollar_volume: shape={dollar_volume.shape}")
print(f"index sample: {prices.index[:3].tolist()}")

# --- Universe mask: top-50 by 30-day trailing dollar ADV, per bar -----
mask = universe.run(prices, dollar_volume, top_n=50, adv_window=720)
n_active = mask.sum(axis=1)
print(f"\nuniverse mask shape: {mask.shape}")
print(
    f"active names per bar -- mean: {n_active.mean():.1f}, "
    f"median: {int(n_active.median())}, min: {int(n_active.min())}, max: {int(n_active.max())}"
)

# --- Survivorship shape: when each asset first appears ----------------
first_valid = prices.apply(lambda s: s.first_valid_index()).sort_values()
print("\nfirst valid bar by asset (oldest 10):")
print(first_valid.head(10).to_string())
print("\nfirst valid bar by asset (newest 10):")
print(first_valid.tail(10).to_string())

# --- Sanity: top-50 at the most recent bar ----------------------------
latest_row = mask.iloc[-1]
latest_active = latest_row[latest_row].index.tolist()
print(f"\ntop-{len(latest_active)} at {mask.index[-1]}:")
print(sorted(latest_active))

## Cell 2 — Stage 2: Frequency / PIT shift

Skill: `backtest_engine2/skills/14-frequency.md`. Apply a 1-bar PIT shift so signal logic sees only data knowable before the current bar. Spec: *Hurst window ends 1 hour before current timestamp.* `freq="intraday"` is passthrough at native (hourly) resolution; `delay=1` does `prices.shift(1)`.

In [ ]:
from backtest.alpha_pipeline import frequency

# 1-bar PIT shift. prices_pit[t] == prices[t-1]; first row is NaN by design.
prices_pit = frequency.run(prices, freq="intraday", delay=1)

# --- Sanity ------------------------------------------------------------
print(f"prices_pit shape: {prices_pit.shape}")
print(f"prices_pit dtype: {prices_pit.dtypes.iloc[0]}")
print(f"first row all-NaN (PIT shift)? {prices_pit.iloc[0].isna().all()}")
print(f"second row all-NaN (only assets without bar 0)? {prices_pit.iloc[1].isna().all()}")

# Spot check: prices_pit[t] should equal prices[t-1] on a known-non-NaN cell.
spot_t = 100
spot_asset = "BTCUSDT"
lhs = prices_pit.iloc[spot_t][spot_asset]
rhs = prices.iloc[spot_t - 1][spot_asset]
print(f"\nspot check at bar {spot_t}, asset {spot_asset}:")
print(f"  prices_pit.iloc[{spot_t}]      = {lhs}")
print(f"  prices.iloc[{spot_t - 1}]        = {rhs}")
print(f"  equal? {lhs == rhs}")

# Magnitude sanity: row counts identical, last bar's PIT value = penultimate raw.
print(f"\nlast raw bar  ({prices.index[-1]}): BTC={prices.iloc[-1]['BTCUSDT']:.2f}")
print(f"last PIT bar  ({prices_pit.index[-1]}): BTC={prices_pit.iloc[-1]['BTCUSDT']:.2f}")
print(f"penultimate raw ({prices.index[-2]}): BTC={prices.iloc[-2]['BTCUSDT']:.2f}")

## Cell 3 — Stage 2.5: Data clamp — SKIPPED

**Decision: skip `data_clamp` for this hunt.**

Trial run at `k=5, window=24, trailing=True` clipped 0.44% of cells across every asset, with max raw/clean ratios of 1.2-1.6x. Inspection of the top-10 clipped events identified them as **real market moves** (DOGE squeeze 2021-01-28, UNI token event 2024-02-23, Chainlink rally onset 2019-06-13, etc.) — not feed glitches. Binance data is clean enough that the cost of suppressing real volatility exceeds the benefit of phantom-glitch hygiene.

The `data_clamp` module remains available for future hunts on noisier feeds. To re-enable here, replace the cell body with:

```python
from backtest.alpha_pipeline import data_clamp
prices_clean = data_clamp.run(prices, k=5, window=24, trailing=True)
prices_pit_clean = prices_clean.shift(1)
```

In [ ]:
# data_clamp SKIPPED. Alias clean variables to the raw inputs so downstream
# cells (signal, evaluate) use a single consistent naming convention.
prices_clean = prices
prices_pit_clean = prices_pit

print("data_clamp SKIPPED -- see Cell 3 heading for rationale.")
print(f"prices_clean is prices?         {prices_clean is prices}")
print(f"prices_pit_clean is prices_pit? {prices_pit_clean is prices_pit}")
print(f"shapes: prices_clean={prices_clean.shape}, prices_pit_clean={prices_pit_clean.shape}")

## Cell 4 — Stage 2.7 EDA: C3 (rolling cointegration) + F1 (liquidity floor)

Skill: `backtest_engine2/skills/14c-eda.md`.

**Sub-claim C3:** *Pairs in the universe are cointegrated under an OLS log-price relationship in a point-in-time sense — not just over the full sample, but in a substantial fraction of rolling 12-month windows.*

**Why it matters:** without cointegration, the spread is not mean-reverting — it's a random walk — and the whole strategy collapses. Spec assumes a tradable cointegrated pool exists at every monthly reselection.

**Pre-committed decision rule:**
- **Supported** if ≥ 20 pairs have stability fraction ≥ 0.6 (cointegrated in at least 60% of rolling monthly windows) AFTER F1.
- **Inconclusive** if 10-19 pairs pass — proceed but flag.
- **Refuted** if < 10 pairs pass → **stop the hunt**.

**Method:**
- Rolling 12-month window (8,760 bars), stepped monthly (720 bars).
- At each step: OLS `log_B ~ log_A` → residuals → ADF with `maxlag=12` → p-value.
- Per-pair stability fraction = `(# windows where p < 0.05) / total windows`.
- F1 post-filter: drop pairs with joint min daily ADV < $1M.
- Parallelized with `joblib` (16 cores available).

In [ ]:
import time
from statsmodels.tsa.stattools import adfuller
from joblib import Parallel, delayed

# --- Config -----------------------------------------------------------
WINDOW_BARS = 8760     # 12 months
STEP_BARS = 720        # monthly step
MIN_OVERLAP = 12960    # 18 months minimum
ADF_MAXLAG = 12
P_THRESHOLD = 0.05
STABILITY_THRESHOLD = 0.6
LIQUIDITY_FLOOR_USD_DAILY = 1_000_000  # $1M/day per leg

# --- Build qualifying pair list --------------------------------------
log_prices_pit = np.log(prices_pit_clean)
assets = log_prices_pit.columns.tolist()
pairs = []
for i, a in enumerate(assets):
    valid_a = log_prices_pit[a].notna()
    for b in assets[i + 1:]:
        overlap = int((valid_a & log_prices_pit[b].notna()).sum())
        if overlap >= MIN_OVERLAP:
            pairs.append((a, b, overlap))
print(f"qualifying pairs (overlap >= {MIN_OVERLAP} bars): {len(pairs)}")

# --- Rolling cointegration test (parallelized) -----------------------
def rolling_coint(a_vals, b_vals, window, step, maxlag):
    """Engle-Granger rolling: returns array of p-values per window."""
    p_values = []
    for end in range(window, len(a_vals) + 1, step):
        start = end - window
        a_win = a_vals[start:end]
        b_win = b_vals[start:end]
        X = np.column_stack([np.ones_like(a_win), a_win])
        coef, *_ = np.linalg.lstsq(X, b_win, rcond=None)
        resid = b_win - coef[0] - coef[1] * a_win
        try:
            p = adfuller(resid, maxlag=maxlag, regression="c")[1]
        except Exception:
            p = np.nan
        p_values.append(p)
    return np.array(p_values)


def process_pair(a, b, log_px_a, log_px_b, vol_a, vol_b, window, step, maxlag):
    """Single-pair computation; runs in a worker process."""
    both = pd.concat([log_px_a, log_px_b], axis=1).dropna()
    if len(both) < window:
        return None
    p_vals = rolling_coint(both.iloc[:, 0].values, both.iloc[:, 1].values, window, step, maxlag)
    p_vals = p_vals[~np.isnan(p_vals)]
    if len(p_vals) == 0:
        return None

    # Joint daily ADV (24 * mean hourly USDT volume on the overlap bars)
    vol_both = pd.concat([vol_a, vol_b], axis=1).dropna()
    joint_adv_daily = float(min(vol_both.iloc[:, 0].mean(), vol_both.iloc[:, 1].mean()) * 24)

    return {
        "asset_a": a,
        "asset_b": b,
        "n_windows": int(len(p_vals)),
        "stability_frac": float((p_vals < P_THRESHOLD).mean()),
        "median_p": float(np.median(p_vals)),
        "min_p": float(p_vals.min()),
        "joint_min_adv_usd_daily": joint_adv_daily,
    }


t0 = time.time()
results_list = Parallel(n_jobs=-1, verbose=0)(
    delayed(process_pair)(
        a, b,
        log_prices_pit[a], log_prices_pit[b],
        dollar_volume[a], dollar_volume[b],
        WINDOW_BARS, STEP_BARS, ADF_MAXLAG,
    )
    for a, b, _overlap in pairs
)
results_list = [r for r in results_list if r is not None]
print(f"rolling cointegration computed in {time.time() - t0:.1f}s for {len(results_list)} pairs")

coint = pd.DataFrame(results_list)

# --- F1: liquidity floor ---------------------------------------------
before = len(coint)
coint_filtered = coint[coint["joint_min_adv_usd_daily"] >= LIQUIDITY_FLOOR_USD_DAILY].copy()
after = len(coint_filtered)
print(
    f"\nF1 filter (joint min ADV >= ${LIQUIDITY_FLOOR_USD_DAILY/1e6:.1f}M daily): "
    f"{before} -> {after} pairs"
)

# --- Distribution of stability ---------------------------------------
stab = coint_filtered["stability_frac"]
print(f"\nstability_frac distribution (after F1):")
for thr in (0.9, 0.8, 0.7, 0.6, 0.5, 0.3):
    print(f"  >= {thr:.1f}: {int((stab >= thr).sum())} pairs")
print(f"  median: {stab.median():.3f}, mean: {stab.mean():.3f}")

# --- Decision rule ---------------------------------------------------
n_pass = int((coint_filtered["stability_frac"] >= STABILITY_THRESHOLD).sum())
if n_pass >= 20:
    verdict = "SUPPORTED"
elif n_pass >= 10:
    verdict = "INCONCLUSIVE"
else:
    verdict = "REFUTED -- stop the hunt"
print(f"\n=== C3+F1 decision: {verdict} ({n_pass} pairs with stability_frac >= {STABILITY_THRESHOLD}) ===")

# --- Top-30 list & save ----------------------------------------------
top = (
    coint_filtered
    .sort_values(["stability_frac", "median_p"], ascending=[False, True])
    .head(30)
    .reset_index(drop=True)
)
display_cols = ["asset_a", "asset_b", "stability_frac", "median_p", "n_windows", "joint_min_adv_usd_daily"]
print(f"\nTop-30 pairs by stability fraction (then ascending median_p):")
with pd.option_context("display.width", 200, "display.max_rows", 30):
    print(top[display_cols].round({"stability_frac": 3, "median_p": 4, "joint_min_adv_usd_daily": 0}).to_string())

top_pairs = list(zip(top["asset_a"], top["asset_b"]))
print(f"\ntop_pairs saved for downstream EDA cells: {len(top_pairs)} pairs")

## Cell 5 — Stage 2.7 EDA: C1 (Hurst distribution on cointegrated spreads)

Skill: `backtest_engine2/skills/14c-eda.md`.

**Sub-claim C1:** *Pair spreads in our cointegrated universe enter anti-persistent regimes (H < 0.5) often enough that the spec's entry gate fires meaningfully.*

**Why it matters:** Cell 4 confirmed long-run cointegration. C1 asks the more granular question — do the *local* dynamics (168h rolling Hurst) actually flag mean-reversion often enough to trade. If H < 0.5 fires 1% of the time, the strategy almost never trades, opportunity cost dominates.

**Pre-committed decision rule:**
- **Supported** if median pair has `fraction(H < 0.5) ≥ 0.20`.
- **Inconclusive** if 0.10–0.20.
- **Refuted** if < 0.10 → strategy barely fires; **stop the hunt**.

**Method:**
- Filter `coint_filtered` to `n_windows >= 30` (defensive: drop short-history pairs from Cell 4 where stability_frac was noisy). Re-rank, take top-30.
- Per pair: rolling vol-scaled hedge ratio `b_t = std(lr_A) / std(lr_B)` over 720 bars (spec Eq. 2), then spread `s_t = log(p_A) − b_t · log(p_B)` (spec Eq. 1).
- Rolling GHE over 168-bar window, `q = 1`, `τ ∈ {1, 2, 4, 8, 16, 32}`, regress `log(K_q(τ))` on `log(τ)`, slope = H (spec Eqs. 3–4).
- Aggregate: per-pair `frac(H<0.5)`, then median across pairs.
- Save `hurst_by_pair: dict[(a,b) → pd.Series]` for C2/C4/C5.

Parallelised across 16 cores.

In [ ]:
import time

# --- Config -----------------------------------------------------------
N_WINDOWS_MIN = 30
TOP_N_PAIRS = 30
HEDGE_WINDOW = 720       # 30 days * 24 hours
HURST_WINDOW = 168       # 7 days * 24 hours
TAU_ARR = np.array([1, 2, 4, 8, 16, 32], dtype=int)  # 2^n for n=0..5
Q = 1

# --- Re-rank with n_windows filter -----------------------------------
coint_n30 = coint_filtered[coint_filtered["n_windows"] >= N_WINDOWS_MIN].copy()
print(f"after n_windows >= {N_WINDOWS_MIN}: {len(coint_n30)} pairs (was {len(coint_filtered)})")

top30 = (
    coint_n30
    .sort_values(["stability_frac", "median_p"], ascending=[False, True])
    .head(TOP_N_PAIRS)
    .reset_index(drop=True)
)
top_pairs = list(zip(top30["asset_a"], top30["asset_b"]))
print(f"top_pairs (n_windows >= {N_WINDOWS_MIN}) updated: {len(top_pairs)} pairs")

# --- GHE on a single window ------------------------------------------
def hurst_window(x: np.ndarray, tau_arr: np.ndarray, q: int = 1) -> float:
    """Generalized Hurst Exponent on a 1D window via spec's K_q(tau) formula."""
    if np.isnan(x).any():
        return np.nan
    abs_xq = np.abs(x) ** q
    denom = abs_xq.mean()
    if denom <= 0 or not np.isfinite(denom):
        return np.nan
    K = np.empty(len(tau_arr), dtype=float)
    for i, tau in enumerate(tau_arr):
        diffs = np.abs(x[tau:] - x[:-tau]) ** q
        K[i] = diffs.mean() / denom
    valid = (K > 0) & np.isfinite(K)
    if valid.sum() < 2:
        return np.nan
    slope, _ = np.polyfit(np.log(tau_arr[valid]), np.log(K[valid]), 1)
    return slope / q


def rolling_hurst(series: pd.Series, window: int, tau_arr: np.ndarray, q: int = 1) -> pd.Series:
    """Rolling GHE on a Series; output aligned to end-of-window timestamps."""
    x = series.values.astype(float)
    n = len(x)
    out = np.full(n, np.nan)
    for end in range(window, n + 1):
        out[end - 1] = hurst_window(x[end - window:end], tau_arr, q)
    return pd.Series(out, index=series.index, name="hurst")


def compute_spread(log_pa: pd.Series, log_pb: pd.Series, hedge_window: int) -> pd.Series:
    """Spec spread: s_t = log(p_A_t) - b_t * log(p_B_t), where b_t = std(lr_A)/std(lr_B) over hedge_window."""
    lr_a = log_pa.diff()
    lr_b = log_pb.diff()
    std_a = lr_a.rolling(hedge_window, min_periods=hedge_window).std()
    std_b = lr_b.rolling(hedge_window, min_periods=hedge_window).std()
    b = std_a / std_b
    return log_pa - b * log_pb


def process_pair(a, b, log_pa, log_pb, hedge_window, hurst_window_size, tau_arr, q):
    """Per-pair: compute spread + rolling Hurst, return summary dict + series."""
    both = pd.concat([log_pa, log_pb], axis=1).dropna()
    if len(both) < hedge_window + hurst_window_size:
        return None
    spread = compute_spread(both.iloc[:, 0], both.iloc[:, 1], hedge_window).dropna()
    if len(spread) < hurst_window_size:
        return None
    h_series = rolling_hurst(spread, hurst_window_size, tau_arr, q)
    valid_h = h_series.dropna()
    if len(valid_h) == 0:
        return None
    return {
        "pair": (a, b),
        "n_bars": int(len(valid_h)),
        "frac_below_0_5": float((valid_h < 0.5).mean()),
        "median_h": float(valid_h.median()),
        "mean_h": float(valid_h.mean()),
        "p10": float(valid_h.quantile(0.10)),
        "p25": float(valid_h.quantile(0.25)),
        "p75": float(valid_h.quantile(0.75)),
        "p90": float(valid_h.quantile(0.90)),
        "hurst_series": h_series,
        "spread_series": spread,
    }


t0 = time.time()
results = Parallel(n_jobs=-1, verbose=0)(
    delayed(process_pair)(
        a, b,
        log_prices_pit[a], log_prices_pit[b],
        HEDGE_WINDOW, HURST_WINDOW, TAU_ARR, Q,
    )
    for a, b in top_pairs
)
results = [r for r in results if r is not None]
print(f"rolling Hurst computed in {time.time() - t0:.1f}s for {len(results)} pairs")

# --- Save Hurst by pair for downstream cells -------------------------
hurst_by_pair = {r["pair"]: r["hurst_series"] for r in results}
spread_by_pair = {r["pair"]: r["spread_series"] for r in results}

# --- Per-pair summary table ------------------------------------------
summary = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ("hurst_series", "spread_series")}
        for r in results
    ]
).sort_values("frac_below_0_5", ascending=False).reset_index(drop=True)
summary["asset_a"] = summary["pair"].apply(lambda x: x[0])
summary["asset_b"] = summary["pair"].apply(lambda x: x[1])

print(f"\nPer-pair Hurst summary (sorted by frac_below_0_5):")
with pd.option_context("display.width", 200, "display.max_rows", 30):
    print(
        summary[["asset_a", "asset_b", "frac_below_0_5", "median_h", "mean_h", "p10", "p25", "p75", "p90", "n_bars"]]
        .round({"frac_below_0_5": 3, "median_h": 3, "mean_h": 3, "p10": 3, "p25": 3, "p75": 3, "p90": 3})
        .to_string()
    )

# --- Aggregate across all pair-bars ----------------------------------
all_h = np.concatenate([r["hurst_series"].dropna().values for r in results])
print(f"\nAggregate H distribution across {len(all_h):,} pair-bars:")
for p in (1, 5, 10, 25, 50, 75, 90, 95, 99):
    print(f"  p{p:2d}: {np.percentile(all_h, p):.3f}")
print(f"\nfraction(H < 0.5) across all pair-bars: {(all_h < 0.5).mean():.3f}")
print(f"median pair's frac(H < 0.5):           {summary['frac_below_0_5'].median():.3f}")
print(f"mean pair's frac(H < 0.5):             {summary['frac_below_0_5'].mean():.3f}")
print(f"min  / max pair frac(H < 0.5):         {summary['frac_below_0_5'].min():.3f} / {summary['frac_below_0_5'].max():.3f}")

# --- Decision ---------------------------------------------------------
med_frac = summary["frac_below_0_5"].median()
if med_frac >= 0.20:
    verdict = "SUPPORTED"
elif med_frac >= 0.10:
    verdict = "INCONCLUSIVE"
else:
    verdict = "REFUTED -- stop the hunt"
print(f"\n=== C1 decision: {verdict} (median pair frac(H<0.5) = {med_frac:.3f}) ===")

## Cell 5b — C1 re-scope: joint cointegration × Hurst-firing score

**Why this cell exists.** Cell 5 ran C1 on Cell 4's cointegration-only top-30 and was **REFUTED** at the strict decision rule (median pair frac(H<0.5) = 0.089). The bimodal pair-level distribution made it clear that long-run cointegration (Cell 4's ranker) and short-run anti-persistence (the spec's entry gate) are nearly *anti-correlated* in our data — JST-X pairs cointegrate but trend; NEXO-X pairs trade-trigger but ranked lower on cointegration.

**Hypothesis update (per `14c-eda.md` Step 4b — partial hold, narrow):** the spec's premise *"cointegrated pairs are anti-persistent"* does not hold universally, but a **subset** of pairs satisfies both. Re-rank by a joint score `stability_frac × frac(H<0.5)` and re-test C1 on the new top-30.

**What this cell will do:**
- Compute rolling Hurst + frac(H<0.5) on **all 622** `n_windows >= 30` cointegrated pairs (~6-13 min, parallelised).
- Build `joint_score = stability_frac * frac(H<0.5)`.
- New top-30 by joint score. Recompute full series only for those (memory-friendly).
- Re-evaluate the C1 decision rule on the new top-30.
- Overwrite `top_pairs`, `hurst_by_pair`, `spread_by_pair` with the new set. Downstream cells (C2, C4, C5) use the new set.

**Pre-committed decision rule (same as Cell 5, now applied to the new set):**
- **Supported** if median pair `frac(H<0.5) ≥ 0.20`.
- **Inconclusive** if 0.10–0.20.
- **Refuted** if < 0.10 → stop the hunt.

In [ ]:
def hurst_summary(a, b, lpa, lpb, hw, hu, tau, q):
    """Summary-only (no series): memory-efficient pass over many pairs."""
    both = pd.concat([lpa, lpb], axis=1).dropna()
    if len(both) < hw + hu:
        return None
    spread = compute_spread(both.iloc[:, 0], both.iloc[:, 1], hw).dropna()
    if len(spread) < hu:
        return None
    h = rolling_hurst(spread, hu, tau, q).dropna()
    if len(h) == 0:
        return None
    return {
        "asset_a": a, "asset_b": b,
        "n_bars": int(len(h)),
        "frac_below_0_5": float((h < 0.5).mean()),
        "median_h": float(h.median()),
        "mean_h": float(h.mean()),
    }


# --- Cache: compute once, reuse on rerun --------------------------------
CACHE_DIR = REPO_ROOT / "backtest_engine2" / "data" / "_cache"
CACHE_DIR.mkdir(exist_ok=True)
JOINT_CACHE = CACHE_DIR / "hunt_2026-06-02_joint_scores.parquet"

if JOINT_CACHE.exists():
    print(f"Loading cached joint scores from {JOINT_CACHE.name}")
    all_summary = pd.read_parquet(JOINT_CACHE)
    print(f"  {len(all_summary)} pairs cached")
else:
    candidate_pairs = list(zip(coint_n30["asset_a"], coint_n30["asset_b"]))
    print(f"Computing Hurst summary on all {len(candidate_pairs)} pairs (no cache)...")
    t0 = time.time()
    all_summaries = Parallel(n_jobs=-1, verbose=0)(
        delayed(hurst_summary)(
            a, b,
            log_prices_pit[a], log_prices_pit[b],
            HEDGE_WINDOW, HURST_WINDOW, TAU_ARR, Q,
        )
        for a, b in candidate_pairs
    )
    all_summaries = [r for r in all_summaries if r is not None]
    print(f"  done in {time.time() - t0:.1f}s for {len(all_summaries)} pairs")
    all_summary = pd.DataFrame(all_summaries).merge(
        coint_n30[["asset_a", "asset_b", "stability_frac", "median_p", "n_windows", "joint_min_adv_usd_daily"]],
        on=["asset_a", "asset_b"], how="left",
    )
    all_summary["joint_score"] = all_summary["stability_frac"] * all_summary["frac_below_0_5"]
    all_summary.to_parquet(JOINT_CACHE)
    print(f"  cached to {JOINT_CACHE.name}")

# --- Per-asset concentration cap (greedy fill) --------------------------
PER_ASSET_CAP = 3  # max pairs containing any single asset

def cap_select(df, k, n_target):
    sorted_df = df.sort_values("joint_score", ascending=False).reset_index(drop=True)
    counts, picked = {}, []
    for _, row in sorted_df.iterrows():
        if len(picked) >= n_target: break
        a, b = row["asset_a"], row["asset_b"]
        if counts.get(a, 0) >= k or counts.get(b, 0) >= k: continue
        picked.append(row)
        counts[a] = counts.get(a, 0) + 1
        counts[b] = counts.get(b, 0) + 1
    return pd.DataFrame(picked).reset_index(drop=True), counts


new_top30, asset_counts = cap_select(all_summary, k=PER_ASSET_CAP, n_target=TOP_N_PAIRS)
new_top_pairs = list(zip(new_top30["asset_a"], new_top30["asset_b"]))
print(f"\nAfter K={PER_ASSET_CAP} per-asset cap, top-{TOP_N_PAIRS} pairs across {len(asset_counts)} unique assets:")
disp = ["asset_a", "asset_b", "joint_score", "stability_frac", "frac_below_0_5", "median_h", "n_windows", "n_bars"]
with pd.option_context("display.width", 200, "display.max_rows", 30):
    print(new_top30[disp].round({"joint_score": 4, "stability_frac": 3, "frac_below_0_5": 3, "median_h": 3}).to_string())

# --- Recompute full series only for the capped top-30 -------------------
print(f"\nRecomputing full Hurst + spread series for top-{len(new_top_pairs)}...")
t0 = time.time()
new_results = Parallel(n_jobs=-1, verbose=0)(
    delayed(process_pair)(
        a, b,
        log_prices_pit[a], log_prices_pit[b],
        HEDGE_WINDOW, HURST_WINDOW, TAU_ARR, Q,
    )
    for a, b in new_top_pairs
)
new_results = [r for r in new_results if r is not None]
print(f"  done in {time.time() - t0:.1f}s for {len(new_results)} pairs")

# --- Overwrite globals ---------------------------------------------------
top_pairs = [r["pair"] for r in new_results]
hurst_by_pair = {r["pair"]: r["hurst_series"] for r in new_results}
spread_by_pair = {r["pair"]: r["spread_series"] for r in new_results}
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("hurst_series", "spread_series")}
    for r in new_results
])
summary["asset_a"] = summary["pair"].apply(lambda x: x[0])
summary["asset_b"] = summary["pair"].apply(lambda x: x[1])

# --- C1 verdict on the capped set ---------------------------------------
med_frac = float(summary["frac_below_0_5"].median())
mean_frac = float(summary["frac_below_0_5"].mean())
min_frac = float(summary["frac_below_0_5"].min())
print(f"\nCapped top-30 frac(H<0.5) stats:")
print(f"  median: {med_frac:.3f}    mean: {mean_frac:.3f}    min: {min_frac:.3f}")
if med_frac >= 0.20: verdict = "SUPPORTED"
elif med_frac >= 0.10: verdict = "INCONCLUSIVE"
else: verdict = "REFUTED -- stop the hunt"
print(f"\n=== C1 (joint score, K={PER_ASSET_CAP} cap) decision: {verdict} (median = {med_frac:.3f}) ===")

# --- Concentration check -------------------------------------------------
ac = pd.Series(asset_counts).sort_values(ascending=False)
print(f"\nAsset concentration (showing all {len(ac)} unique assets):")
print(ac.to_string())

## Cell 6 — Stage 2.7 EDA: C2 (mean-reversion conditional on H<0.5)

Skill: `backtest_engine2/skills/14c-eda.md`.

**Sub-claim C2:** *When `H < 0.5` AND the spread is in the 1-2σ entry band, the spread mean-reverts over the next 24-72 hours. The Hurst gate isn't just statistically anti-persistent — it's predictively useful.*

**Why this is the most important EDA cell so far:** C1 + Cell 5b showed the gate **fires often** on the K=3-capped top-30. C2 asks whether the gate fires *correctly*. If H<0.5 has zero predictive power for spread reversion, the entire strategy is built on noise — no exit calibration can rescue it.

**Pre-committed decision rule:**
- **Supported** if median pair satisfies BOTH:
  - `IC(z, fwd_return | H<0.5) < −0.02` at h=24h (negative IC = above-mean spreads tend to fall; the prediction works).
  - `IC | H<0.5 < IC | H≥0.5` (the gate adds predictive value over baseline; H<0.5 isn't just inheriting weak reversion that exists everywhere).
- **Inconclusive** if one but not both, or |IC| < 0.02.
- **Refuted** if median IC is non-negative or wrong sign → **stop the hunt**, signal is noise.

**Method:**
- For each of the K=3-capped 30 pairs:
  - `z_t = (spread_t − rolling_mean_168h_t) / rolling_std_168h_t`.
  - `fwd_t(h) = spread_{t+h} − spread_t` for `h ∈ {24, 72}`.
  - IC = Pearson correlation of `z` vs `fwd`, split by `H<0.5` vs `H≥0.5`.
- Side diagnostic: naive PnL of `pnl = −sign(z) · fwd` on the entry-condition bars (`|z|∈[1,2] ∧ H<0.5`). No exits, no costs — pure entry-signal edge.
- Aggregate medians + per-pair table.

In [ ]:
Z_WINDOW = 168          # spec TW
HORIZONS = (24, 72)     # h=24h (one-day) and h=72h (matches spec vertical barrier)
IC_THRESHOLD = -0.02    # median IC under H<0.5 must beat this (more negative) to "support"


def analyze_pair_c2(pair, spread, hurst, z_window=Z_WINDOW, horizons=HORIZONS):
    a, b = pair
    m = spread.rolling(z_window, min_periods=z_window).mean()
    sd = spread.rolling(z_window, min_periods=z_window).std()
    z = (spread - m) / sd

    result = {"pair": pair, "asset_a": a, "asset_b": b}
    for h in horizons:
        fwd = spread.shift(-h) - spread
        df = pd.concat([z.rename("z"), hurst.rename("hurst"), fwd.rename("fwd")], axis=1).dropna()
        h_low_mask = df["hurst"] < 0.5
        h_high_mask = ~h_low_mask

        ic_low = (
            df.loc[h_low_mask, ["z", "fwd"]].corr().iloc[0, 1]
            if h_low_mask.sum() > 10 else np.nan
        )
        ic_high = (
            df.loc[h_high_mask, ["z", "fwd"]].corr().iloc[0, 1]
            if h_high_mask.sum() > 10 else np.nan
        )

        # Naive PnL on entry condition (no cost, no exit calibration)
        in_band = (df["z"].abs() > 1) & (df["z"].abs() < 2)
        entry = in_band & h_low_mask
        n_entries = int(entry.sum())
        if n_entries > 10:
            pnl = -np.sign(df.loc[entry, "z"]) * df.loc[entry, "fwd"]
            mean_pnl = float(pnl.mean())
            win_rate = float((pnl > 0).mean())
        else:
            mean_pnl = win_rate = np.nan

        result[f"ic_low_h{h}"]   = float(ic_low)  if pd.notna(ic_low)  else np.nan
        result[f"ic_high_h{h}"]  = float(ic_high) if pd.notna(ic_high) else np.nan
        result[f"n_entries_h{h}"]= n_entries
        result[f"mean_pnl_h{h}"] = mean_pnl
        result[f"win_rate_h{h}"] = win_rate
    return result


t0 = time.time()
c2_results = Parallel(n_jobs=-1, verbose=0)(
    delayed(analyze_pair_c2)(p, spread_by_pair[p], hurst_by_pair[p])
    for p in top_pairs
)
print(f"C2 computed in {time.time() - t0:.1f}s for {len(c2_results)} pairs")

c2 = pd.DataFrame(c2_results)

# --- Per-pair table (sort by ic_low_h24 ascending: most-negative IC first) -
disp_cols = [
    "asset_a", "asset_b",
    "ic_low_h24", "ic_high_h24",
    "ic_low_h72", "ic_high_h72",
    "n_entries_h24", "mean_pnl_h24", "win_rate_h24",
]
print("\nPer-pair C2 results (sorted by ic_low_h24 ascending; more-negative = stronger reversion):")
with pd.option_context("display.width", 200, "display.max_rows", 30):
    print(
        c2.sort_values("ic_low_h24")[disp_cols]
          .round({"ic_low_h24": 4, "ic_high_h24": 4, "ic_low_h72": 4, "ic_high_h72": 4,
                  "mean_pnl_h24": 6, "win_rate_h24": 3})
          .to_string()
    )

# --- Aggregates ---------------------------------------------------------
print("\nMedian IC across pairs:")
for h in HORIZONS:
    med_low = c2[f"ic_low_h{h}"].median()
    med_high = c2[f"ic_high_h{h}"].median()
    delta = med_low - med_high
    print(f"  h={h:>3}h:  IC|H<0.5 = {med_low:+.4f}    IC|H>=0.5 = {med_high:+.4f}    delta = {delta:+.4f}")

print("\nIC|H<0.5 distribution at h=24:")
for thr in (-0.10, -0.05, -0.02, 0.0, 0.02, 0.05):
    sign = ">" if thr >= 0 else "<"
    print(f"  pairs with ic_low_h24 {sign} {thr:+.2f}: {(c2['ic_low_h24'] < thr).sum() if thr < 0 else (c2['ic_low_h24'] > thr).sum()}")

print("\nNaive PnL summary at h=24 (no cost, no exit calibration):")
print(f"  median mean_pnl across pairs: {c2['mean_pnl_h24'].median():.6f}")
print(f"  median win_rate across pairs: {c2['win_rate_h24'].median():.3f}")
print(f"  pairs with mean_pnl > 0:      {(c2['mean_pnl_h24'] > 0).sum()} / {len(c2)}")
print(f"  pairs with win_rate > 0.50:   {(c2['win_rate_h24'] > 0.50).sum()} / {len(c2)}")

# --- Decision rule ------------------------------------------------------
med_ic_low = c2["ic_low_h24"].median()
med_ic_high = c2["ic_high_h24"].median()
cond_a = med_ic_low < IC_THRESHOLD
cond_b = med_ic_low < med_ic_high
if cond_a and cond_b:
    verdict = "SUPPORTED"
elif cond_a or cond_b:
    verdict = "INCONCLUSIVE"
else:
    verdict = "REFUTED -- stop the hunt"
print(f"\n=== C2 decision: {verdict} ===")
print(f"  rule (a) median IC|H<0.5 < {IC_THRESHOLD}: {med_ic_low:+.4f} -> {cond_a}")
print(f"  rule (b) median IC|H<0.5 < IC|H>=0.5:     {med_ic_low:+.4f} < {med_ic_high:+.4f} -> {cond_b}")

## Hypothesis-update — End of Stage 2.7 EDA

**Original hypothesis** (from `strategy_engine2/output/hurst-optimal-exit-pairs/spec.json`):
> When a crypto pair's log-spread is anti-persistent (local generalized Hurst `H < 0.5`) and the spread sits 1-2σ from its rolling mean, the spread mean-reverts. Per-trade O-U fit calibrates profit-take and stop-loss thresholds; 72h vertical barrier.

### Sub-claim verdicts

| Sub-claim | Verdict | Key number |
|---|---|---|
| **C1** — anti-persistent regimes fire often | SUPPORTED on K=3-capped joint-score top-30 | median pair `frac(H<0.5)` = 0.45 |
| **C2** — `H<0.5` predicts mean reversion | REFUTED raw (IC at h=24 = +0.034, wrong direction); **BORDERLINE refined** (MEDIUM gate + OU exits: win 0.68, TRAIN PnL +0.54σ, VAL ≈ 0) | per-pair PnL bimodal; see _tmp scripts |
| **C3** — pairs are persistently cointegrated | SUPPORTED | 33 pairs ≥ 60% rolling stability |
| **C4** — entry band populated | implicit ✓ | 216k entry events across 30 pairs |
| **C5** — OU calibration stable | implicit ✓ | median φ=0.97, σ=0.008 |
| **C6** — TP not barrier-dominated | SUPPORTED | TP 60-66%, barrier 13-21% (with OU exits) |
| **C7** — cross-pair diversification | **SUPPORTED** | median pair-pair PnL corr = **−0.001**; PCA: 14 PCs for 50% variance (vs 15 random) — within noise of fully independent |
| F1 — liquidity floor | non-binding | top-50 by ADV all above $1M/day |

### Net status: partial hold, narrowed

The spec **as written** is REFUTED in our data — the binary `H<0.5` gate doesn't reliably predict spread reversion (IC test). The spec works in **TRAIN, fails marginally in VAL** under naive PnL with hold-to-horizon. With OU-calibrated exits, win rate jumps to 68% but VAL mean PnL is at break-even.

The hunt proceeds with a **refined version** of the spec:
- **Pair selection:** K=3-capped joint score (`stability_frac × frac(H<0.5)`) — Cell 5b top-30. NOT cointegration alone.
- **Entry gate (MEDIUM):** `H ≤ 0.42` AND `time_in_band ≤ 6` AND `|z| ∈ [1.0, 1.33]` AND original `1 < |z| < 2` AND `H < 0.5`. Reduces entries 10× but improves reversion-completion P from 43% → 60% at h=72.
- **Exit:** per-spec O-U calibration on rolling 168h, daily anchor approximation, TP at π+\* / SL at π−\* / 72h barrier.
- **Universe:** 30 pairs across 26 unique assets (K=3 cap). Diversified per C7.

### Observations beyond the spec — candidates for future hunts

1. **NEXO single-asset mean-reversion** is strong: `frac(H<0.5) = 0.82` on NEXO's own log-price. This is a cleaner, simpler alpha than the pair-spread strategy.
2. **Cointegration ≠ anti-persistence at hourly scale** in crypto. JST-X pairs cointegrate (long-run) but trend (short-run). NEXO-X pairs are reverse. The joint score is needed.
3. **Refined entry gate** (Hurst-quintile + time-in-band + magnitude) materially improves direction prediction over the binary spec gate. Generalisable to other mean-reversion strategies.
4. **OU optimizer prefers very loose stops** (median π−\* = −10σ, grid edge) for high-φ pairs — the math says hold through drawdowns. Worth noting for OU-based strategies broadly.
5. **|z| velocity / acceleration** weakly predict completion (Q5 wins ~52% vs Q1 ~48%) — could be useful filters in other reversion strategies.

### Decision: proceed to Stage 3 (signal_construction) with the narrowed version

Path **(β)** — implement the refined strategy as a stateful custom signal module, route through engine Stage 23 (CPCV + DSR) for the real verdict. The naive-PnL refute is *not* the verdict — Stage 23 with proper costs + cross-validation is.

## Cell 7 — Build PIT cache for HurstPairsStrategy

Pre-compute the per-pair time-series the Strategy class needs for PIT monthly pair selection and per-trade OU calibration. **Built once, cached, loaded instantly on subsequent runs.**

Two cached parquets:
- `_cache/hurst_pairs_coint_series.parquet` — rolling cointegration p-values per (pair, anchor_t). Long format.
- `_cache/hurst_pairs_hurst_series.parquet` — rolling Hurst + spread per (pair, bar_t). Long format. All 622 qualifying pairs (not just top-30 — strategy may pick different pairs per CPCV fold).

Both are PIT by construction: every series cell at timestamp `t` was computed from data `≤ t` via a trailing rolling window.

First run ≈ 21 min (parallelized across 16 cores). Subsequent runs near-instant.

In [ ]:
import time
from statsmodels.tsa.stattools import adfuller

CACHE_DIR_C7 = REPO_ROOT / "backtest_engine2" / "data" / "_cache"
CACHE_DIR_C7.mkdir(exist_ok=True, parents=True)
COINT_SERIES_CACHE = CACHE_DIR_C7 / "hurst_pairs_coint_series.parquet"
HURST_SERIES_CACHE = CACHE_DIR_C7 / "hurst_pairs_hurst_series.parquet"

WINDOW_BARS = 8760    # 12-month rolling cointegration window
STEP_BARS   = 720     # monthly step
MIN_OVERLAP = 12960   # 18 months minimum joint history
ADF_MAXLAG  = 12
HEDGE_W     = 720
HURST_W     = 168
TAU_ARR_C7  = np.array([1, 2, 4, 8, 16, 32])
Q_C7        = 1


if COINT_SERIES_CACHE.exists() and HURST_SERIES_CACHE.exists():
    print(f"Loading cached PIT series from disk...")
    coint_series_df = pd.read_parquet(COINT_SERIES_CACHE)
    hurst_series_df = pd.read_parquet(HURST_SERIES_CACHE)
else:
    print(f"Building PIT cache (~21 min on first run)...")

    log_pp = np.log(prices_pit_clean)
    assets_c7 = log_pp.columns.tolist()
    qualifying_pairs_c7 = []
    for i, a in enumerate(assets_c7):
        va = log_pp[a].notna()
        for b in assets_c7[i + 1:]:
            ov = int((va & log_pp[b].notna()).sum())
            if ov >= MIN_OVERLAP:
                qualifying_pairs_c7.append((a, b, ov))
    print(f"  {len(qualifying_pairs_c7)} qualifying pairs (overlap >= {MIN_OVERLAP})")

    # --- Cointegration series per pair ---
    def _rolling_coint_series(av, bv, w, s, ml):
        out = []
        for end in range(w, len(av) + 1, s):
            a_w = av[end - w:end]; b_w = bv[end - w:end]
            X = np.column_stack([np.ones_like(a_w), a_w])
            c, *_ = np.linalg.lstsq(X, b_w, rcond=None)
            r = b_w - c[0] - c[1] * a_w
            try:
                p = adfuller(r, maxlag=ml, regression="c")[1]
            except Exception:
                p = np.nan
            out.append((end - 1, p, float(c[1])))
        return out

    def _proc_coint(a, b, lpa, lpb):
        both = pd.concat([lpa, lpb], axis=1).dropna()
        if len(both) < WINDOW_BARS:
            return []
        ts = both.index
        res = _rolling_coint_series(
            both.iloc[:, 0].values, both.iloc[:, 1].values,
            WINDOW_BARS, STEP_BARS, ADF_MAXLAG,
        )
        return [
            {"pair_id": f"{a}|{b}", "asset_a": a, "asset_b": b,
             "anchor_t": ts[idx], "p_value": p, "hedge_ratio": br}
            for idx, p, br in res
        ]

    t0 = time.time()
    coint_chunks = Parallel(n_jobs=-1, verbose=0)(
        delayed(_proc_coint)(a, b, log_pp[a], log_pp[b])
        for a, b, _ in qualifying_pairs_c7
    )
    coint_flat = [item for chunk in coint_chunks for item in chunk]
    coint_series_df = pd.DataFrame(coint_flat)
    print(f"  cointegration series: {time.time() - t0:.1f}s ({len(coint_series_df):,} rows)")
    coint_series_df.to_parquet(COINT_SERIES_CACHE)

    # --- Hurst + spread series per pair ---
    def _hurst_win(x, tau, q=1):
        if np.isnan(x).any(): return np.nan
        abs_xq = np.abs(x) ** q
        denom = abs_xq.mean()
        if denom <= 0 or not np.isfinite(denom): return np.nan
        K = np.empty(len(tau))
        for i, t in enumerate(tau):
            K[i] = (np.abs(x[t:] - x[:-t]) ** q).mean() / denom
        v = (K > 0) & np.isfinite(K)
        if v.sum() < 2: return np.nan
        s, _ = np.polyfit(np.log(tau[v]), np.log(K[v]), 1)
        return s / q

    def _rolling_hurst(s, w, tau, q=1):
        x = s.values.astype(float)
        n = len(x); out = np.full(n, np.nan)
        for end in range(w, n + 1):
            out[end - 1] = _hurst_win(x[end - w:end], tau, q)
        return pd.Series(out, index=s.index)

    def _compute_spread(lpa, lpb, hw):
        lr_a = lpa.diff(); lr_b = lpb.diff()
        return lpa - (lr_a.rolling(hw, min_periods=hw).std() / lr_b.rolling(hw, min_periods=hw).std()) * lpb

    def _proc_hurst(a, b, lpa, lpb):
        both = pd.concat([lpa, lpb], axis=1).dropna()
        if len(both) < HEDGE_W + HURST_W:
            return None
        spread = _compute_spread(both.iloc[:, 0], both.iloc[:, 1], HEDGE_W).dropna()
        if len(spread) < HURST_W:
            return None
        h = _rolling_hurst(spread, HURST_W, TAU_ARR_C7, Q_C7)
        df = pd.DataFrame({
            "pair_id": f"{a}|{b}", "asset_a": a, "asset_b": b,
            "bar_t": spread.index, "hurst": h.values, "spread": spread.values,
        })
        df = df.dropna(subset=["hurst"])
        return df

    t0 = time.time()
    hurst_chunks = Parallel(n_jobs=-1, verbose=0)(
        delayed(_proc_hurst)(a, b, log_pp[a], log_pp[b])
        for a, b, _ in qualifying_pairs_c7
    )
    hurst_chunks = [c for c in hurst_chunks if c is not None]
    hurst_series_df = pd.concat(hurst_chunks, ignore_index=True)
    print(f"  hurst+spread series: {time.time() - t0:.1f}s ({len(hurst_series_df):,} rows)")
    hurst_series_df.to_parquet(HURST_SERIES_CACHE)

# Summary
print(f"\nCache state:")
print(f"  cointegration series: {len(coint_series_df):,} rows, {coint_series_df['pair_id'].nunique()} pairs")
print(f"  hurst+spread series:  {len(hurst_series_df):,} rows, {hurst_series_df['pair_id'].nunique()} pairs")
print(f"  median coint anchors per pair: {int(coint_series_df.groupby('pair_id').size().median())}")
print(f"  median hurst bars per pair:    {int(hurst_series_df.groupby('pair_id').size().median())}")
print(f"  coint p-value range: [{coint_series_df['p_value'].min():.4f}, {coint_series_df['p_value'].max():.4f}]")
print(f"  hurst range:         [{hurst_series_df['hurst'].min():.3f}, {hurst_series_df['hurst'].max():.3f}]")

## Cell 8 — Define + instantiate HurstPairsStrategy

Skill: `backtest_engine2/skills/02-strategy.md`.

Implementation lives in `backtest_engine2/experiments/hurst_pairs_strategy.py` (one file, ~250 lines). Highlights:

- `rebalance_frequency = "h"` (check every hour).
- `fit(data)` — initial PIT pair selection using `data.t` (the train-window end).
- `generate_weights(data, t)` — state machine:
  - On month boundary: re-select universe; force-close de-listed pairs.
  - For each pair in current universe: lookup pre-computed `spread/hurst/z/abs_z/sign/tib/hedge_b` at `t`; if in trade → check TP/SL/72h exits; else check refined gate → on fire, calibrate OU per-trade, open.
- Gross-normalised weights output (sum |·| = 1 per bar).

This cell instantiates the strategy (does NOT run the engine yet — Cell 10 does). Validates: `__repr__`, rebalance count, required_data, sanity initial pair selection at the latest possible date.

In [ ]:
import sys
NB_DIR = REPO_ROOT / "backtest_engine2" / "experiments"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

from hurst_pairs_strategy import HurstPairsStrategy

strategy = HurstPairsStrategy(
    prices_pit=prices_pit_clean,
    coint_series_df=coint_series_df,
    hurst_series_df=hurst_series_df,
    # all other params at defaults: gate_params, OU paths=1000, K=3, top_n=30
)

print(strategy)  # validates __repr__
print(f"required_data: {strategy.required_data()}")

# Rebalance count sanity (hourly => one per bar in dates)
hourly_dates = prices_pit_clean.index
rebalance_dates = strategy.rebalance_dates(hourly_dates)
print(f"\nrebalance schedule: {len(rebalance_dates):,} bars (out of {len(hourly_dates):,} total)")
print(f"  first: {rebalance_dates[0]}")
print(f"  last:  {rebalance_dates[-1]}")
print(f"  per year (hourly 24/7): {len(rebalance_dates) / (len(hourly_dates) / (24*365)):.0f}  (should be ~8760)")

# Sanity: initial PIT pair selection at the latest available timestamp
latest_t = prices_pit_clean.index[-1]
init_pairs = strategy._select_pairs_pit(latest_t)
print(f"\nPIT pair selection at {latest_t} (latest available):")
print(f"  selected {len(init_pairs)} pairs (target {strategy.top_n}, K={strategy.per_asset_cap} cap)")
print(f"  unique assets: {len({a for ab in init_pairs for a in ab})}")
asset_cnt = pd.Series([a for ab in init_pairs for a in ab]).value_counts()
print(f"  top 8 asset counts: {dict(asset_cnt.head(8))}")
print(f"\n  first 10 pairs by joint score:")
for p in init_pairs[:10]:
    print(f"    {p[0]:>12} - {p[1]:<12}")

## Cell 10 — Stage 4: Engine single-path sanity run

Skill: `backtest_engine2/skills/04-costs.md` + `05-engine-single-path.md` + `23-engine-onramp.md`.

- Panel built from **raw prices** (not `prices_pit` — engine uses actual prices for mark-to-market); `volume_shares = dollar_volume / prices` (skill 04 §5.3 says panel volume must be share volume); `universe = mask` from Cell 1; `FieldSpec(lag=0)` since strategy uses pre-cached PIT data.
- **CostModel**: `Commission(bps=10)` + `Spread(half_bps=5)` + `ShortBorrow(annual_bps=200, trading_days=8760)`. ~20 bps round-trip per leg + 2% annual on shorts.
- **LiquidityCap(cap_pct=0.10, adv_lookback=168)**: 10% of weekly ADV.
- **HurstPairsStrategy(verbose=True, assert_pit=True)**: monthly progress prints + PIT assertions firing at every pair selection and OU calibration.
- **Engine** runs single-path on full 2019-2026 history. No `train_dates` — strategy bootstraps via first month boundary inside `generate_weights`.
- This is **NOT the verdict** — single path, no DSR. Sanity check that the strategy runs cleanly, produces coherent equity, costs/turnover are in plausible ranges, and PIT assertions pass.

**Compute estimate:** ~25-30 min (bottleneck: ~25k OU calibrations × ~50ms each).

In [ ]:
from backtest.data.panel import DataPanel, FieldSpec
from backtest.costs import Commission, Spread, ShortBorrow, CompositeCostModel, LiquidityCap
from backtest.engine.engine import Engine

# Convert dollar_volume to share_volume (panel expects shares, not dollars; skill 04 §5.3)
volume_shares = (dollar_volume / prices).replace([np.inf, -np.inf], np.nan)

panel = DataPanel(
    prices=prices,
    volume=volume_shares,
    universe=mask,
    specs={"prices": FieldSpec(lag=0)},  # strategy uses pre-cached PIT, no double-shift
)
print(f"panel built: dates={len(panel.dates):,}, assets={len(panel.assets_all)}, has_volume={panel.has_field('volume')}")

costs = CompositeCostModel([
    Commission(bps=10),
    Spread(half_bps=5),
    ShortBorrow(annual_bps=200, trading_days=24 * 365),  # hourly 24/7 crypto
])
liq = LiquidityCap(cap_pct=0.10, adv_lookback=168)
print(f"cost components: {[type(m).__name__ for m in costs.models]}")
print(f"liquidity: cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")

# Re-instantiate strategy with verbose=True for progress prints
strategy_v = HurstPairsStrategy(
    prices_pit=prices_pit_clean,
    coint_series_df=coint_series_df,
    hurst_series_df=hurst_series_df,
    verbose=True,
    assert_pit=True,
)
print(f"\nstrategy: {strategy_v}")

engine = Engine(
    data=panel,
    strategy=strategy_v,
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000.0,
)
print(f"\nengine built — starting run (no train_dates; strategy bootstraps from first month boundary)")
print(f"  expected runtime: ~25-30 min")

t0 = time.time()
result = engine.run()
print(f"\n>>> engine.run() done in {(time.time() - t0) / 60:.1f} min")

# Summary
print(f"\n=== BACKTEST RESULT ===")
print(f"  bars run:       {len(result.equity_curve):,}")
print(f"  start equity:   ${result.equity_curve.iloc[0]:,.0f}")
print(f"  end equity:     ${result.equity_curve.iloc[-1]:,.0f}")
print(f"  total return:   {(result.equity_curve.iloc[-1] / result.equity_curve.iloc[0] - 1) * 100:+.2f}%")
print(f"  total costs:    ${result.costs.sum():,.0f}")
print(f"  costs / initial: {result.costs.sum() / result.equity_curve.iloc[0] * 100:.2f}%")

# Engine summary metrics
report = result.summary(ann_factor=24 * 365)
print(f"\n  --- MetricsReport (annualised to 8760/year) ---")
for k in ("sharpe", "ann_return", "max_drawdown", "turnover", "margin", "hit_rate", "pct_profitable_days"):
    if hasattr(report, k):
        v = getattr(report, k)
        if v is not None:
            print(f"  {k:>22}: {v}")

# Strategy counters
print(f"\n  --- Strategy counters ---")
print(f"  total entries opened: {strategy_v._n_entries_total:,}")
print(f"  exits by TP:          {strategy_v._n_exits_tp:,}")
print(f"  exits by SL:          {strategy_v._n_exits_sl:,}")
print(f"  exits by barrier:     {strategy_v._n_exits_barrier:,}")
print(f"  exits by de-listing:  {strategy_v._n_exits_delisted:,}")
print(f"  active at end:        {len(strategy_v.active_trades):,}")